# Milestone 2 — Computer Vision

Task: predict product category from image.

Compare small CNN (scratch) vs MobileNetV2 transfer learning.

In [ ]:
import os
import sys
from pathlib import Path

for path in [
    Path("/content/smart-product-intelligence"),
    Path("/content/drive/MyDrive/smart-product-intelligence"),
    Path.cwd(),
    Path.cwd().parent,
]:
    if (path / "src" / "data.py").exists():
        os.chdir(path)
        if str(path) not in sys.path:
            sys.path.insert(0, str(path))
        print("Project root:", path)
        break

import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image

from src.data import artifacts_dir, load_image_manifest, load_splits
from src.models import build_small_cnn, build_transfer_model, set_global_seed
from src.utils import plot_confusion_matrix, plot_learning_curves, save_json, setup_notebook_path

setup_notebook_path()
set_global_seed(42)

In [ ]:
IMG_SIZE = (128, 128)
BATCH = 32
EPOCHS = 8

manifest = load_image_manifest()
manifest = manifest[manifest["image_available"] == True].copy()
splits = load_splits()
train_ids = set(splits["train_products"]["product_id"])
val_ids = set(splits["val_products"]["product_id"])
test_ids = set(splits["test_products"]["product_id"])

manifest["split"] = manifest["product_id"].map(
    lambda pid: "train" if pid in train_ids else ("val" if pid in val_ids else "test")
)
manifest = manifest.merge(
    splits["train_products"][["product_id", "category_leaf"]],
    on="product_id",
    how="left",
)

labels = sorted(manifest["category_leaf"].dropna().unique())
label_map = {label: idx for idx, label in enumerate(labels)}
manifest["label"] = manifest["category_leaf"].map(label_map)
manifest = manifest.dropna(subset=["label"])
print(f"Samples: {len(manifest)} | Classes: {len(labels)}")

In [ ]:
def load_image(path):
    img = Image.open(path).convert("RGB").resize(IMG_SIZE)
    return np.array(img, dtype=np.float32)

def make_dataset(frame, training=False):
    paths = frame["local_image_path"].tolist()
    labels_arr = frame["label"].astype(int).tolist()

    def _gen():
        for p, y in zip(paths, labels_arr):
            try:
                arr = load_image(p)
                if training and np.random.rand() < 0.5:
                    arr = tf.image.flip_left_right(arr).numpy()
                yield arr, y
            except Exception:
                continue

    return tf.data.Dataset.from_generator(
        _gen,
        output_signature=(
            tf.TensorSpec(shape=(*IMG_SIZE, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int32),
        ),
    ).batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(manifest[manifest["split"] == "train"], training=True)
val_ds = make_dataset(manifest[manifest["split"] == "val"])
test_ds = make_dataset(manifest[manifest["split"] == "test"])

In [ ]:
cnn = build_small_cnn(num_classes=len(labels))
cnn_hist = cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
cnn_eval = cnn.evaluate(test_ds, verbose=0)
print("Scratch CNN test:", dict(zip(cnn.metrics_names, cnn_eval)))

In [ ]:
transfer = build_transfer_model(num_classes=len(labels), backbone="mobilenetv2")
transfer_hist = transfer.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
transfer_eval = transfer.evaluate(test_ds, verbose=0)
print("Transfer test:", dict(zip(transfer.metrics_names, transfer_eval)))

plot_learning_curves(transfer_hist, "Transfer MobileNetV2", artifacts_dir() / "m2_transfer_curves.png")

ckpt = artifacts_dir().parent / "checkpoints"
ckpt.mkdir(parents=True, exist_ok=True)
transfer.save(ckpt / "vision_transfer.keras")
with open(ckpt / "vision_label_map.json", "w", encoding="utf-8") as f:
    json.dump(label_map, f, indent=2)

save_json(
    {
        "scratch_cnn": dict(zip(cnn.metrics_names, map(float, cnn_eval))),
        "transfer_mobilenetv2": dict(zip(transfer.metrics_names, map(float, transfer_eval))),
    },
    artifacts_dir() / "vision_metrics.json",
)